# TrackPulse — Bronze Capture to Delta (Incremental Job)

**Purpose:** Process newly available Event Hubs Capture Avro files into Streaming Bronze Delta.

**Schedule:** Run from ADF before the Silver incremental job. The existing Auto Loader checkpoint must be preserved.

**Do not:** reset/delete the checkpoint, change its path, overwrite Bronze, or run the historical transformation notebook here.

ADF executes all cells in this notebook from top to bottom.

In [0]:
# Purpose: Define independent paths for streaming ingestion.
# Safety: Configuration only; does not read or write data.

raw_capture_path = (
    "abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/"
    "ns-trackpulse-dev/telemetry-events/"
)

bronze_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "bronze/telemetry_stream/"
)

stream_checkpoint_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "checkpoints/databricks_streaming/telemetry_capture/"
)

stream_schema_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "checkpoints/schemas/telemetry_capture/"
)

print("Raw source:", raw_capture_path)
print("Bronze target:", bronze_path)
print("Streaming checkpoint:", stream_checkpoint_path)
print("Auto Loader schema:", stream_schema_path)

Raw source: abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/ns-trackpulse-dev/telemetry-events/
Bronze target: abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/bronze/telemetry_stream/
Streaming checkpoint: abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/checkpoints/databricks_streaming/telemetry_capture/
Auto Loader schema: abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/checkpoints/schemas/telemetry_capture/


In [0]:
# Purpose: Incrementally discover Event Hubs Capture Avro files.
# Safety: Defines a streaming DataFrame only; does not start a streaming query.

from pyspark.sql.functions import col, decode, from_json, current_timestamp
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    LongType, BooleanType
)

event_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("vehicle_id", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("speed_kph", DoubleType(), True),
    StructField("odometer_km", DoubleType(), True),
    StructField("rpm", LongType(), True),
    StructField("fuel_rate_lph", DoubleType(), True),
    StructField("engine_temp_c", DoubleType(), True),
    StructField("ignition_on", BooleanType(), True),
    StructField("trip_id", StringType(), True),
    StructField("shipment_id", StringType(), True),
    StructField("driver_id", StringType(), True),
    StructField("route_id", StringType(), True),
    StructField("cargo_temp_c", DoubleType(), True)
])

capture_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "avro")
    .option("cloudFiles.schemaLocation", stream_schema_path)
    .load(raw_capture_path)
)

parsed_stream = (
    capture_stream
    .withColumn("event_json", decode(col("Body"), "UTF-8"))
    .withColumn("event", from_json(col("event_json"), event_schema))
    .select(
        col("SequenceNumber").alias("capture_sequence_number"),
        col("EnqueuedTimeUtc").alias("capture_enqueued_time_utc"),
        col("event.*"),
        col("event_json")
    )
)

print("Streaming source defined.")

Streaming source defined.


In [0]:
# Purpose: Exclude test events and append production telemetry to Bronze Delta.
# Safety: Uses a dedicated checkpoint; never resets or overwrites existing checkpoints.

from pyspark.sql.functions import coalesce, lit, to_timestamp

production_stream = (
    parsed_stream
    .filter(col("event_id").isNotNull())
    .filter(~col("event_id").startswith("TEST-"))
    .withColumn(
        "event_timestamp_utc",
        to_timestamp(col("event_timestamp"))
    )
    .withColumn("bronze_ingested_at", current_timestamp())
    .withColumn("source_format", lit("event_hubs_capture_avro"))
)

query = (
    production_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", stream_checkpoint_path)
    .trigger(availableNow=True)
    .start(bronze_path)
)

query.awaitTermination()
print("Available Capture files processed. Check query status and Bronze counts.")

Available Capture files processed. Check query status and Bronze counts.


In [0]:
# Purpose: Validate Bronze Delta ingestion and check for test events.
# Safety: Read-only validation; does not modify Delta data or checkpoints.

from pyspark.sql.functions import col

bronze_df = (
    spark.read
    .format("delta")
    .load(bronze_path)
)

total_records = bronze_df.count()

test_records = (
    bronze_df
    .filter(col("event_id").startswith("TEST-"))
    .count()
)

print(f"Total Bronze records: {total_records:,}")
print(f"Test records found: {test_records:,}")
print("Bronze schema:")
bronze_df.printSchema()

display(bronze_df.limit(10))

Total Bronze records: 167,522
Test records found: 0
Bronze schema:
root
 |-- capture_sequence_number: long (nullable = true)
 |-- capture_enqueued_time_utc: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- speed_kph: double (nullable = true)
 |-- odometer_km: double (nullable = true)
 |-- rpm: long (nullable = true)
 |-- fuel_rate_lph: double (nullable = true)
 |-- engine_temp_c: double (nullable = true)
 |-- ignition_on: boolean (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- shipment_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- cargo_temp_c: double (nullable = true)
 |-- event_json: string (nullable = true)
 |-- event_timestamp_utc: timestamp (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 

capture_sequence_number,capture_enqueued_time_utc,event_id,event_timestamp,vehicle_id,latitude,longitude,speed_kph,odometer_km,rpm,fuel_rate_lph,engine_temp_c,ignition_on,trip_id,shipment_id,driver_id,route_id,cargo_temp_c,event_json,event_timestamp_utc,bronze_ingested_at,source_format
10,10/25/2026 8:56:44 PM,VEH-01001-0002000,2026-10-02T09:19:02Z,VEH-01001,29.916718,32.165056,50.1,339756.994,1997,4.948,94.34,true,TRIP-01001-001,SHP-01001,DRV-00281,RTE-005,-0.61,"{""event_id"":""VEH-01001-0002000"",""event_timestamp"":""2026-10-02T09:19:02Z"",""vehicle_id"":""VEH-01001"",""latitude"":29.916718,""longitude"":32.165056,""speed_kph"":50.1,""odometer_km"":339756.994,""rpm"":1997,""fuel_rate_lph"":4.948,""engine_temp_c"":94.34,""ignition_on"":true,""trip_id"":""TRIP-01001-001"",""shipment_id"":""SHP-01001"",""driver_id"":""DRV-00281"",""route_id"":""RTE-005"",""cargo_temp_c"":-0.61}",2026-10-02T09:19:02.000Z,2026-10-28T13:36:52.241Z,event_hubs_capture_avro
11,10/25/2026 8:56:44 PM,VEH-01002-0002000,2026-10-02T09:19:02Z,VEH-01002,29.329311,30.83815,42.08,223419.276,1927,8.331,93.25,true,TRIP-01002-001,SHP-01002,DRV-00281,RTE-006,null,"{""event_id"":""VEH-01002-0002000"",""event_timestamp"":""2026-10-02T09:19:02Z"",""vehicle_id"":""VEH-01002"",""latitude"":29.329311,""longitude"":30.83815,""speed_kph"":42.08,""odometer_km"":223419.276,""rpm"":1927,""fuel_rate_lph"":8.331,""engine_temp_c"":93.25,""ignition_on"":true,""trip_id"":""TRIP-01002-001"",""shipment_id"":""SHP-01002"",""driver_id"":""DRV-00281"",""route_id"":""RTE-006"",""cargo_temp_c"":null}",2026-10-02T09:19:02.000Z,2026-10-28T13:36:52.241Z,event_hubs_capture_avro
12,10/25/2026 8:56:44 PM,VEH-01003-0002000,2026-10-02T09:19:02Z,VEH-01003,30.067355,31.238801,38.16,437581.221,1745,3.737,93.6,true,TRIP-01003-001,SHP-01003,DRV-00281,RTE-010,null,"{""event_id"":""VEH-01003-0002000"",""event_timestamp"":""2026-10-02T09:19:02Z"",""vehicle_id"":""VEH-01003"",""latitude"":30.067355,""longitude"":31.238801,""speed_kph"":38.16,""odometer_km"":437581.221,""rpm"":1745,""fuel_rate_lph"":3.737,""engine_temp_c"":93.6,""ignition_on"":true,""trip_id"":""TRIP-01003-001"",""shipment_id"":""SHP-01003"",""driver_id"":""DRV-00281"",""route_id"":""RTE-010"",""cargo_temp_c"":null}",2026-10-02T09:19:02.000Z,2026-10-28T13:36:52.241Z,event_hubs_capture_avro
13,10/25/2026 8:56:44 PM,VEH-01004-0002000,2026-10-02T09:19:02Z,VEH-01004,29.983199,31.137029,29.28,272599.3,1509,4.087,86.12,true,TRIP-01004-001,SHP-01004,DRV-00282,RTE-001,-0.27,"{""event_id"":""VEH-01004-0002000"",""event_timestamp"":""2026-10-02T09:19:02Z"",""vehicle_id"":""VEH-01004"",""latitude"":29.983199,""longitude"":31.137029,""speed_kph"":29.28,""odometer_km"":272599.3,""rpm"":1509,""fuel_rate_lph"":4.087,""engine_temp_c"":86.12,""ignition_on"":true,""trip_id"":""TRIP-01004-001"",""shipment_id"":""SHP-01004"",""driver_id"":""DRV-00282"",""route_id"":""RTE-001"",""cargo_temp_c"":-0.27}",2026-10-02T09:19:02.000Z,2026-10-28T13:36:52.241Z,event_hubs_capture_avro
14,10/25/2026 8:56:44 PM,VEH-01005-0002000,2026-10-02T09:19:02Z,VEH-01005,30.199356,31.225277,40.13,144691.056,1904,7.588,90.21,true,TRIP-01005-001,SHP-01005,DRV-00282,RTE-010,null,"{""event_id"":""VEH-01005-0002000"",""event_timestamp"":""2026-10-02T09:19:02Z"",""vehicle_id"":""VEH-01005"",""latitude"":30.199356,""longitude"":31.225277,""speed_kph"":40.13,""odometer_km"":144691.056,""rpm"":1904,""fuel_rate_lph"":7.588,""engine_temp_c"":90.21,""ignition_on"":true,""trip_id"":""TRIP-01005-001"",""shipment_id"":""SHP-01005"",""driver_id"":""DRV-00282"",""route_id"":""RTE-010"",""cargo_temp_c"":null}",2026-10-02T09:19:02.000Z,2026-10-28T13:36:52.241Z,event_hubs_capture_avro
15,10/25/2026 8:56:44 PM,VEH-01006-0002000,2026-10-02T09:19:02Z,VEH-01006,30.027134,31.219993,50.61,298305.964,2082,4.315,90.72,true,TRIP-01006-001,SHP-01006,DRV-00282,RTE-006,null,"{""event_id"":""VEH-01006-0002000"",""event_timestamp"":""2026-10-02T09:19:02Z"",""vehicle_id"":""VEH-01006"",""latitude"":30.027134,""lo